# COM 763: Advanced Machine Learning (Task 1)
# SQL Query Runtime Prediction

**Author:** Aled Caio Rowlands  
**Module:** COM 763 Advanced Machine Learning

---

### Abstract

This report documents the development of a machine learning pipeline for SQL query runtime prediction using the quereis from a text to SQL benchmark and measured SQLite execution runtime and features. The project began with a binary classification framing, asking whether a query would be fast or slow, but that model was weak in both the data and the feature set. I therefore shifted the main framing toward regression on `log(runtime_seconds)` .

Across the project, the most important finding was not a strong headline score but a clear limitation: structural SQL features alone do not transfer well across unseen databases. Cross-schema classification remained near-random, and cross-schema regression produced negative `R2` values. Later experiments showed why: runtime is strongly shaped by schema-level factors such as row counts and index coverage, which are invisible in the SQL text (this may seem obvious but it was only revealed in this project rhoguh trail and erro). When database-level schema statistics were added and evaluation was restricted to known schemas indiviudal databases, performance improved sharply for some indexed databases, reaching `R2 = 0.945` on a `debit_card_specializing` database and `R2 = 0.929` on a `formula_1` database, while still failing on large poorly indexed schemas such as `financial`.

I could have found a dataset on kagge adn finetuned it or took inspiraation of of previous peoples hard wokr. However I ambistiously tried to solve the world, whilst failing each time ,this however helped to reduce scope increase quality and create a small but well designed final end product. This report is therefore a record of iterative development, debugging, and model reframing. Rather than claiming a universal runtime predictor, I show exactly where the pipeline worked, where it failed, and what those failures reveal about the problem. Furhtermore the project went from: 
How to make a universal small model of runtime prediciton that transfer across databases
->
Would the same method work on a multiple seen databases
->
Focusing on one database 
->
Focusing on one database with schema understanding

Furthermor many bugs, lack of EDA and lack of knowldege in how to correctly train models an dthe types of machine learnign used made many fialures occur. However thorugh narrowing the scope of the porject and moving trhough the issues, a workable streamlit app was produced.

## 1. Problem Definition and System Framing

The real-world motivation for this project is straightforward. SQL query performance is difficult to judge from the text of a query alone, yet poor runtime behaviour has real costs. In production systems, slow queries consume compute time, delay dashboards and pipelines, increase cloud spend, and force developers or DBAs to spend time profiling queries manually. A lightweight machine learning model that could estimate runtime or at least warn that a query is likely to be expensive would therefore have clear practical value.

I intentionally focused on a constrained version of that problem: can runtime be predicted from the **structure of the SQL query itself**, without using deep integration with the target database? That framing was attractive because it would make deployment simple. In principle, a user could paste a query and their database into this model and receive a rough estimate before running it.

The central research question became:

> **Can a model trained on queries from some databases predict runtime on databases it has never seen before?**

That question matters because true usefulness depends on cross-schema transfer. If a model only works on schemas it has already seen, then every new deployment would require retraining and the tool would lose much of its value. The research question and machine learnign was adapted and scope reduced as the porject progressed.

It ended up as: 
> **How well can a model trained on a smal quanitty of queries + databse specific info predict runtime on that same database?**

The project did not keep a single formulation throughout. I originally framed it as binary classification (`fast` versus `slow`) because that looked simple to explain and easy to deploy as a warning system. Later evidence showed that this framing was masking key problems. As a result of differing database size and poor distbuion of said queires. The final formulation therefore became supervised regression on `log(runtime_seconds)`.

Several limitations were not recognised early and became more impactful as the project progressed. All timings were collected on my laptop, using SQLite, under one local execution environment, so the model does not learn a hardware-independent law of runtime. The SQL-only feature set also excludes row counts, indexes, and cache state, even though these often drive execution time. That trade-off was deliberate at the start, but later results of course showed that it was also the main reason transfer failed.


## 2. Data Pipeline and Feature Handling

The data source was **BIRD Mini-Dev**, a text-to-SQL benchmark covering 11 SQLite databases with varied domains, including finance, sport, education, and gaming. I chose it because it offers realistic SQL rather than toy classroom examples, spans multiple schemas, and is still small enough to run repeatedly on local hardware. Although it may seem stragne to us ea TExt-to-SQL benchmark, it's use case was intresting for AI to go from NLP to a exectuable accurate SQL qeuiry it provide a foundation for well desinged and labeled databases and well curated and labelled types of quires.

Each query label was produced by actually executing the SQL against its corresponding SQLite database, usually with **3 timing runs** and a **30-second timeout**, and then taking the **median runtime**. This was more expensive than using logs or heuristics, but it meant that every label was a measured observation rather than an inferred estimate.

| Stage | Raw rows | Labelled rows | Fast | Slow |
|------|----------|---------------|------|------|
| Initial pipeline | 425 | 320 | 213 | 107 |
| Data expansion | 498 | 374 | 249 | 125 |

A major methodological decision in the early classifier pipeline was **quantile-based bucketing**: the bottom 50% of runtimes were labelled `fast`, the top 25% were labelled `slow`, and the middle 25% were dropped as ambiguous. This made the classes look cleaner, but it also removed **124 queries (24.9%)** from the labelled dataset. That later turned out to be a structural weakness, because it removed the borderline examples that would help a model learn the transition between fast and slow behaviour. 
![Figure 12 -- Query counts per database](fig12_query_counts_per_db.png)
*Figure 12. Query counts per database. Data volume is highly uneven across schemas.*

The per-database breakdown immediately revealed imbalance problems.

| Database | Queries | Fast | Slow | % Slow |
|------|------:|------:|------:|------:|
| superhero | 50 | 50 | 0 | 0% |
| card_games | 50 | 7 | 43 | 86% |
| student_club | 47 | 47 | 0 | 0% |
| european_football_2 | 45 | 8 | 37 | 82% |
| formula_1 | 41 | 38 | 3 | 7% |
| codebase_community | 35 | 3 | 32 | 91% |
| financial | 32 | 28 | 4 | 12% |
| thrombosis_prediction | 27 | 27 | 0 | 0% |
| toxicology | 25 | 24 | 1 | 4% |
| debit_card_specializing | 19 | 14 | 5 | 26% |
| california_schools | 3 | 3 | 0 | 0% |

These counts show that the dataset is not just small overall; it is uneven inside individual schemas as well. Some databases contain no slow queries at all, while others contain almost nothing but slow queries. That proved to be central to the later debugging story. This was unforrtualy found quite late after many archtecural foundation of this porject surrounding this database. Realsing that I did a lakc of EDA early on which would have prevented hours of effort on a poorly designed classes 

![Figure 4 -- The dropped middle bracket](fig4_dropped_middle_pie.png)
*Figure 4. The quantile-labelling method removed nearly a quarter of the collected data.*

![Figure 5 -- Runtime distribution](fig5_runtime_distribution.png)
*Figure 5. Runtime is heavily right-skewed, which motivated regression on the log scale.*

![Figure 3 -- Label skew per database](fig3_per_db_label_skew.png)
*Figure 3. Label skew varies sharply by schema, showing that labels encode database identity as well as query complexity.*

Feature engineering started with **25 SQL structural features** extracted from the query text. I grouped them into six families so that the feature design remained interpretable rather than arbitrary.

| Family | Examples | Rationale |
|------|------|------|
| Size | `n_tokens`, `query_length` | Longer queries often imply more operations |
| Join complexity | `n_joins`, `n_tables_approx` | More joins can increase intermediate result cost |
| Clause presence | `has_group_by`, `has_order_by`, `has_limit`, `has_union` | Clauses add execution work |
| Subquery and nesting | `n_subqueries`, `has_subquery`, `max_nesting_depth`, `has_correlated_subquery` | Deeply nested logic is often harder to execute |
| Aggregation | `n_count`, `n_sum`, `n_avg`, `n_aggregations` | Aggregations can require scans and grouping |
| Predicates | `n_where_predicates`, `has_between`, `has_in_clause`, `has_exists` | Predicate structure influences filtering cost |

Later phases extended this set in two directions. First, I added **9 EXPLAIN QUERY PLAN features** as a `Tree+Global` variant, including plan step count, scan nodes, search nodes, temporary B-tree sorts, and correlated subquery indicators. Second, in Phase 6, I added **6 schema statistics features**: number of tables, total rows, largest table size, total indexes, index coverage, and log total rows.

This progression reflects the most important lesson from the data pipeline: SQL structure alone was a reasonable starting point, but the deeper the experiments went, the clearer it became that runtime cannot be understood without at least some schema context.


## 3. Model Implementation and Debugging

This section carried the most weight in the project because the report was not supposed to present a single polished model as if it had worked immediately. The important part was the sequence of modelling decisions, the evidence that some of them were wrong, and the debugging logic that followed from the results.

### Phase 1: Unseen-database classification

I started with a cross-schema classifier because it matched the original practical question well: can the model warn that a query will be slow on a database it has never seen before? I held out `financial` and `formula_1` as unseen test databases and trained on the other nine. Four classifier families were explored across the project stages: Random Forest, XGBoost, Gradient Boosting, and Logistic Regression.

![Figure 1 -- Seen vs unseen performance](fig1_seen_vs_unseen_commits.png)
*Figure 1. Seen-DB versus unseen-DB performance across pipeline stages. The gap is the transfer problem.*

![Figure 11 -- All classifier models compared](fig11_all_models_comparison.png)
*Figure 11. No classifier solved the unseen-schema problem in a meaningful way.*

![Figure 6 -- Unseen confusion matrix](fig6_confusion_matrix_unseen.png)
*Figure 6. The classifier misses most slow queries on the unseen holdout.*

| Pipeline stage | Best model | Unseen F1 | Unseen ROC-AUC | Unseen Accuracy |
|------|------|------:|------:|------:|
| Baseline | XGBoost | 0.1765 | 0.5442 | 0.4909 |
| Pipeline improved | XGBoost | 0.2069 | 0.5148 | 0.5490 |
| Data expanded | XGBoost | 0.1739 | 0.4567 | 0.4795 |

These results were the first major warning sign. Unseen `F1` peaked at just `0.2069`, and final unseen `ROC-AUC` dropped to `0.4567`, which is worse than a coin flip. Precision on the slow class was also extremely poor on both holdout databases. That meant that when the model predicted a slow query on an unseen schema, it was usually wrong.

### Phase 2: Seen-database performance and the misleading improvement story

After unseen transfer failed, I asked whether the model had learned anything at all. The natural diagnostic was within-database evaluation: train on some queries from each schema and test on other queries from the same schemas. At first sight, the seen-database results looked better, but they turned out to be misleading.

![Figure 2 -- Seen-DB F1 decline](fig2_seen_f1_decline.png)
*Figure 2. Seen-DB F1 declined as the pipeline improved and more data was added. That is the wrong direction for genuine learning.*

| Pipeline stage | Best model | Seen F1 | Seen ROC-AUC | Seen Accuracy |
|------|------|------:|------:|------:|
| Baseline | Random Forest | 0.5000 | 0.6863 | 0.6765 |
| Pipeline improved | XGBoost | 0.4444 | 0.6701 | 0.6378 |
| Data expanded | Random Forest | 0.3913 | 0.6400 | 0.6410 |

The key debugging insight here was that **performance fell as the pipeline improved and more data was added**. That is the opposite of what should happen if the model is learning a stable signal. I concluded that the earlier seen-database numbers were probably inflated by tiny test sets and outlier-sensitive class metrics, not by real generalisation.

### Phase 3: Switching from classification to regression

At this point I decided that the classifier was part of the problem. It could only say `fast` or `slow`, and its metrics were too sensitive to threshold choice and small test-set composition. I therefore moved to regression on `log(runtime_s)` so that the model had to predict continuous runtime rather than a binary bucket.

![Figure 7 -- Unseen regression scatter](fig7_regression_unseen_scatter.png)
*Figure 7. Predicted versus actual log runtime on the unseen holdout. The model fails to follow the diagonal.*

| Model | MAE (log) | RMSE (log) | R2 (log) | MAE (s) |
|------|------:|------:|------:|------:|
| Linear Regression | 2.6036 | 3.6613 | -2.0381 | 12333.98 |
| Ridge (a=1) | 2.5696 | 3.5528 | -1.8607 | 5553.84 |
| Ridge (a=10) | **2.3846** | **3.0053** | **-1.0469** | **61.98** |
| Lasso | 2.4342 | 3.1485 | -1.2467 | 231.10 |

This was the moment when the project became much clearer. All `R2` values were negative, meaning the model was worse than simply predicting the training mean for every query. Regression made the earlier classifier weakness visible rather than hiding it behind thresholded scores.

### Phase 4 and Phase 5: forensic debugging of the data and the strongest available holdout

Once both classification and regression had failed on unseen databases, I stopped assuming the model was at fault and looked more closely at the data itself. Two issues became clear: first, the dropped middle quartile had removed nearly a quarter of the observations; second, the slow-class proportion varied wildly by schema, from `0%` to `91%`. That meant the labels partly encoded database identity rather than just query complexity.

To test whether the problem could be rescued on the strongest available data, I held out the two databases with 50 queries each: `superhero` and `card_games`. I compared structural SQL features alone, matched structural features, and `Tree+Global` features that included EXPLAIN-plan statistics.

![Figure 8 -- Regression on superhero and card_games](fig8_regression_50q_scatter.png)
*Figure 8. Predictions remain clustered in a narrow band even on the largest available holdout databases.*

![Figure 9 -- Feature coefficients](fig9_feature_coefficients.png)
*Figure 9. Counter-intuitive coefficient signs are a strong sign of schema confounding rather than genuine complexity learning.*

The most damaging example came from `card_games`, where queries taking `0.72s` to `0.94s` were predicted at just `0.001s` to `0.004s`. One of the worst misses was labelled `simple` difficulty, which showed that the structural SQL representation was simply not seeing the reason for slowness. That failure was not due to a lack of tuning. It was a feature limitation.

### Phase 6: adding schema statistics and changing the evaluation question

The final major debugging step was to accept what the earlier phases were telling me: SQL text alone was not enough. I therefore added six database-level schema statistics and changed the evaluation question from cross-schema transfer to **within-database explanation**. Instead of asking whether the model can generalise immediately to a totally new schema, I asked whether it can explain runtime once some schema context is known.

That was not just a final experiment. It was the direct outcome of the debugging process. The whole project moved from “build a universal predictor” to “understand why universal transfer fails and under which conditions runtime becomes predictable.”


## 4. Experimental Evaluation and Model Selection

The evaluation strategy had to match the evolving research question. I therefore used two levels of evaluation.

1. **Cross-schema holdout**, where `financial` and `formula_1` were held out entirely. This tests the hardest and most practically interesting scenario: deployment on a new database.
2. **Within-database evaluation**, where each schema is split internally. This tests whether runtime can be predicted once the model already knows the schema.

Across the project I reported `F1`, `ROC-AUC`, and accuracy for classification, then `MAE`, `RMSE`, and `R2` for regression. That mix of metrics mattered because no single score was enough to explain what the model was actually doing.

### Cross-schema verdict

Across both classification and regression, the verdict was clear: **the SQL-only model did not transfer to unseen schemas in a practically useful way**.

![Figure 10 -- R2 comparison across regression experiments](fig10_r2_comparison.png)
*Figure 10. All unseen-schema regression results remain below `R2 = 0`, confirming that naive mean prediction outperforms the learned model.*

The best SQL-only unseen regression model was Ridge with stronger regularisation (`a = 10`), but even that reached only `R2 = -1.0469` on the `financial + formula_1` holdout. This is not good enough for deployment as a universal predictor.

### Within-database schema-aware results

Once schema statistics were introduced and each database was trained and tested independently, a more nuanced pattern emerged.

![Figure 13 -- Best within-database R2 values](fig13_within_db_r2.png)
*Figure 13. Within-database performance varies sharply by schema. Some databases become highly predictable, while others remain resistant.*

![Figure 14 -- Schema complexity versus R2](fig14_schema_vs_r2.png)
*Figure 14. Index coverage predicts model success much more clearly than raw database size.*

| Database | n | Slow % | Index coverage | Best model | R2 (log) | MAE (log) | MAE (s) |
|------|------:|------:|------:|------|------:|------:|------:|
| debit_card_specializing | 19 | 26% | 0.67 | Ridge(a=10) | **0.945** | 0.446 | 0.031 |
| formula_1 | 41 | 7% | 0.50 | Ridge(a=10) | **0.929** | 0.696 | 0.152 |
| student_club | 47 | 0% | 0.88 | Ridge(a=10) | **0.640** | 0.172 | 0.000049 |
| european_football_2 | 45 | 82% | 0.62 | Lasso | **0.148** | 1.133 | 0.120 |
| superhero | 50 | 0% | 0.00 | Lasso | 0.099 | 0.500 | 0.000106 |
| toxicology | 25 | 4% | 1.00 | RF | 0.009 | 0.723 | 0.001 |
| thrombosis_prediction | 27 | 0% | 0.33 | GBM | -0.141 | 0.644 | 0.001 |
| card_games | 50 | 86% | 0.29 | RF | -0.329 | 1.310 | 0.195 |
| codebase_community | 35 | 91% | 0.38 | RF | -0.390 | 0.363 | 0.230 |
| financial | 32 | 12% | 0.00 | RF | **-10.435** | 1.706 | 0.007 |

This was the strongest piece of evaluation evidence in the entire project. It showed that the feature set was **not universally useless**; instead, its usefulness depended heavily on schema conditions. Databases with moderate to high index coverage and enough meaningful variation were predictable. Databases with zero or very low index coverage remained unpredictable because runtime was dominated by full table scans, I/O timing, and schema-specific behaviour that query text alone could not reveal.

### Final model selection

A single universal final model would be misleading, so my final selection is deliberately nuanced.

- For the **original cross-schema SQL-only objective**, no model met a convincing success threshold. The strongest SQL-only baseline was **Random Forest** for within-database regression and **XGBoost** for classification, but neither solved transfer.
- For the **best-performing schema-aware within-database setting**, the strongest and most consistent model was **Ridge Regression with stronger regularisation (`a = 10`)** once schema statistics were added.

I therefore selected **Ridge(a=10) with schema statistics** as the best final analytical model in the report, because it gave the clearest evidence that runtime becomes predictable once the model is allowed to see the schema context that actually drives cost. I did not select Random Forest as the final answer because its apparent SQL-only advantage disappeared when the project moved to the harder and more honest question of schema transfer.

The most important evaluation conclusion is therefore not “this model wins”, but rather:

> **Cross-schema SQL-only runtime prediction failed, but schema-aware within-database prediction was partially successful, especially on indexed databases.**


## Deployment

A short Streamlit deployment was kept as part of the end-to-end pipeline because the assignment brief requires a working system, not just offline analysis. The app was intended to let a user paste a SQL query, trigger feature extraction, and receive a predicted runtime estimate. In the wider project, it also served as a way to demonstrate that the pipeline outputs could be packaged into an interactive interface rather than remaining as scripts and CSV files.

At the same time, the report findings force an important caveat. The deployment should be interpreted as a proof of system integration, not as evidence that the runtime predictions are universally reliable. The timings were collected on my laptop in SQLite, and the best results depended strongly on schema properties that are not available in a pure SQL-text-only deployment. The Streamlit component therefore demonstrates engineering completion more than it demonstrates universal predictive validity.

- **Streamlit deployment URL:** `https://<your-app-name>.streamlit.app`
- **GitHub repository URL:** `<your-github-repository-link>`


## Conclusion

This project began with a simple question: can SQL query runtime be predicted from the structure of the query alone? By the end of the work, I had a more precise answer.

The answer is **no, not in a robust cross-schema sense**. Structural SQL features alone were not enough to support reliable transfer to unseen databases. Classification results were weak, regression results were worse than a naive mean baseline, and the model repeatedly failed on schemas whose runtime behaviour was driven by factors that the SQL text could not reveal.

However, the project did not end in total failure. The later schema-aware experiments showed that runtime becomes much more predictable when the model knows something about the database it is operating on, especially index coverage. In that sense, the project moved from trying to prove a universal predictor to identifying the exact conditions under which prediction becomes feasible.

Personally, the most useful part of the project was the way the evidence forced me to keep changing the task rather than protecting the original idea. I began with a classifier because it looked neat and practical. I then changed to regression because the classifier was hiding too much. After that, I stopped assuming the model was the only problem and interrogated the data and labels more seriously. Finally, I accepted that schema information was not optional background detail but the core missing signal. That progression taught me more than a single successful score would have done.

The final conclusion is therefore honest and evidence-based: a lightweight SQL-only runtime predictor was not sufficient for unseen-schema deployment, but the project still produced a clear empirical contribution by showing why it failed and by demonstrating that schema-aware within-database modelling can work well under the right conditions.


## References

Li, J., Hui, B., Qu, G., Yang, J., Li, B., Li, B., Wang, B., Qin, B., Geng, R., Huo, N., Zhou, X., Ma, C., Huang, R., Lou, Q., Chen, Z., Zhang, Z., Li, Z., Zhu, J., Cai, T., Chen, R., Chen, X., Huang, S., Liu, K. and Zhu, Y. (2024). *Can LLM Already Serve as A Database Interface? A Big Bench for Large-Scale Database Grounded Text-to-SQLs.* Advances in Neural Information Processing Systems (NeurIPS), 36. Available at: https://arxiv.org/abs/2305.03111

Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., Blondel, M., Prettenhofer, P., Weiss, R., Dubourg, V., Vanderplas, J., Passos, A., Cournapeau, D., Brucher, M., Perrot, M. and Duchesnay, E. (2011). *Scikit-learn: Machine Learning in Python.* Journal of Machine Learning Research, 12, pp.2825-2830.

Breiman, L. (2001). *Random Forests.* Machine Learning, 45(1), pp.5-32.

Friedman, J.H. (2001). *Greedy function approximation: a gradient boosting machine.* Annals of Statistics, 29(5), pp.1189-1232.

Hoerl, A.E. and Kennard, R.W. (1970). *Ridge regression: biased estimation for nonorthogonal problems.* Technometrics, 12(1), pp.55-67.

Marcus, R., Negi, P., Mao, H., Zhang, C., Alizadeh, M., Kraska, T., Papaemmanouil, O. and Tatbul, N. (2019). *Neo: A Learned Query Optimizer.* Proceedings of the VLDB Endowment, 12(11), pp.1705-1718.

Streamlit Inc. (2024). *Streamlit Documentation.* Available at: https://docs.streamlit.io
